# IM-CBGT — entrenamiento repetido con 120 sujetos

Notebook autónomo para Kaggle. Usa exclusivamente los 120 sujetos definidos en `folds.pkl`:

- 60 Control y 60 ADHD.
- `v28p` se incluye porque pertenece a las particiones.
- `v36p` queda fuera automáticamente porque no pertenece a `folds.pkl`.
- 10 seeds × 5 folds.
- Selección de características ajustada únicamente con train en cada fold.
- Guardado de checkpoints, predicciones por ventana y resultados por sujeto.
- Reanudación automática de folds incompletos.


In [1]:
# ============================================================
# 1. IMPORTS, RUTAS Y CONFIGURACIÓN
# ============================================================

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import gc
import json
import pickle
import random
import shutil
import warnings
from collections import Counter
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.signal import welch
from scipy.stats import entropy, kurtosis, skew
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

DATA_ROOT = Path(
    "/kaggle/input/datasets/daprosero/mi-tdah-dataset/"
    "MI_TDAH_Dataset/TDAH"
)
FOLDS_PATH = DATA_ROOT / "folds.pkl"
ADHD_DIR = DATA_ROOT / "ieee" / "ADHD_group"
CONTROL_DIR = DATA_ROOT / "ieee" / "Control_group"

SAVE_ROOT = Path(
    "/kaggle/working/resultados_imcbgt_tdah_ARTICULO_120subjects"
)
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "IMCBGT"
DATABASE_NAME = "TDAH"
SEEDS = list(range(10))
N_FOLDS = 5
EPOCHS = 100
BATCH_SIZE = 16
NUM_WORKERS = 0

# False permite reanudar y reutilizar folds terminados.
# Para borrar y repetir absolutamente todo, cambia a True.
FORCE_RETRAIN = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ARGS = {
    "nb_classes": 2,
    "fs": 128,
    "pca_var": 0.95,
    "chi_k": 100,
    "conv_filters": 64,
    "kernel_size": 3,
    "lstm_units": 64,
    "gru_units": 64,
    "att_heads": 2,
    "dense_units": 128,
    "fusion_units": 256,
    "dropout_rate": 0.5,
}

TRAIN_CONFIG = {
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "scheduler_factor": 0.5,
    "scheduler_patience": 10,
    "scheduler_min_lr": 1e-6,
    "early_stopping_patience": 25,
    "min_delta": 1e-4,
}

print("Device:", DEVICE)
print("Folds:", FOLDS_PATH)
print("ADHD:", ADHD_DIR)
print("Control:", CONTROL_DIR)
print("Resultados:", SAVE_ROOT)


Device: cuda
Folds: /kaggle/input/datasets/daprosero/mi-tdah-dataset/MI_TDAH_Dataset/TDAH/folds.pkl
ADHD: /kaggle/input/datasets/daprosero/mi-tdah-dataset/MI_TDAH_Dataset/TDAH/ieee/ADHD_group
Control: /kaggle/input/datasets/daprosero/mi-tdah-dataset/MI_TDAH_Dataset/TDAH/ieee/Control_group
Resultados: /kaggle/working/resultados_imcbgt_tdah_ARTICULO_120subjects


In [2]:
# ============================================================
# 2. SEMILLAS Y COHORTE EXACTA DEFINIDA POR folds.pkl
# ============================================================

def set_seed(seed, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:
            try:
                torch.use_deterministic_algorithms(True)
            except Exception:
                pass


def normalize_subject_id(value):
    if isinstance(value, bytes):
        value = value.decode("utf-8")
    value = Path(str(value)).name
    if value.lower().endswith(".mat"):
        value = value[:-4]
    return value.strip()


def normalize_folds(raw_folds):
    normalized = []

    for fold_index, fold_item in enumerate(raw_folds):
        if isinstance(fold_item, (list, tuple)) and len(fold_item) == 3:
            train_subjects, val_subjects, test_subjects = fold_item
        elif isinstance(fold_item, dict):
            train_subjects = (
                fold_item.get("train")
                or fold_item.get("train_subjects")
                or fold_item.get("subjects_train")
            )
            val_subjects = (
                fold_item.get("val")
                or fold_item.get("validation")
                or fold_item.get("val_subjects")
                or fold_item.get("subjects_val")
            )
            test_subjects = (
                fold_item.get("test")
                or fold_item.get("test_subjects")
                or fold_item.get("subjects_test")
            )
            if train_subjects is None or val_subjects is None or test_subjects is None:
                raise ValueError(f"No se pudo interpretar el fold {fold_index}.")
        else:
            raise TypeError(
                f"Formato de fold no reconocido en fold {fold_index}: "
                f"{type(fold_item)}"
            )

        normalized.append(
            (
                [normalize_subject_id(x) for x in train_subjects],
                [normalize_subject_id(x) for x in val_subjects],
                [normalize_subject_id(x) for x in test_subjects],
            )
        )

    return normalized


def load_and_validate_folds(folds_path=FOLDS_PATH):
    if not folds_path.exists():
        raise FileNotFoundError(f"No existe folds.pkl:\n{folds_path}")

    with open(folds_path, "rb") as file:
        folds = normalize_folds(pickle.load(file))

    if len(folds) != N_FOLDS:
        raise ValueError(
            f"Se esperaban {N_FOLDS} folds y se encontraron {len(folds)}."
        )

    protocol_subjects = set()
    test_counter = Counter()

    for fold_index, (train_subjects, val_subjects, test_subjects) in enumerate(folds):
        train_set = set(train_subjects)
        val_set = set(val_subjects)
        test_set = set(test_subjects)

        expected_counts = (76, 20, 24)
        current_counts = (len(train_set), len(val_set), len(test_set))
        if current_counts != expected_counts:
            raise ValueError(
                f"Fold {fold_index}: se esperaba train/val/test={expected_counts} "
                f"y se obtuvo {current_counts}."
            )

        if train_set & val_set or train_set & test_set or val_set & test_set:
            raise ValueError(f"Fold {fold_index}: hay solapamiento entre particiones.")

        fold_union = train_set | val_set | test_set
        if len(fold_union) != 120:
            raise ValueError(
                f"Fold {fold_index}: la unión tiene {len(fold_union)} sujetos, no 120."
            )

        protocol_subjects.update(fold_union)
        test_counter.update(test_subjects)

    if len(protocol_subjects) != 120:
        raise ValueError(
            f"folds.pkl define {len(protocol_subjects)} sujetos únicos, no 120."
        )

    invalid_test_counts = {
        subject: count
        for subject, count in test_counter.items()
        if count != 1
    }
    if invalid_test_counts:
        raise ValueError(
            "Cada sujeto debe aparecer exactamente una vez en test: "
            f"{invalid_test_counts}"
        )

    if "v28p" not in protocol_subjects:
        raise ValueError("v28p no aparece en folds.pkl y debería estar incluido.")

    if "v36p" in protocol_subjects:
        raise ValueError("v36p aparece en folds.pkl, pero debería estar excluido.")

    return folds, sorted(protocol_subjects)


def build_mat_index(directory):
    directory = Path(directory)
    if not directory.exists():
        raise FileNotFoundError(f"No existe la carpeta:\n{directory}")

    index = {
        normalize_subject_id(path.stem): path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() == ".mat"
    }
    if not index:
        raise FileNotFoundError(f"No se encontraron .mat en:\n{directory}")
    return index


def load_eeg_mat(mat_path, subject_id, expected_channels=19):
    mat = scipy.io.loadmat(mat_path)

    if subject_id in mat:
        raw = np.asarray(mat[subject_id])
        variable_name = subject_id
    else:
        candidates = []
        for key, value in mat.items():
            if key.startswith("__"):
                continue
            array = np.asarray(value)
            if (
                array.ndim == 2
                and np.issubdtype(array.dtype, np.number)
                and expected_channels in array.shape
            ):
                candidates.append((key, array))

        if not candidates:
            available = {
                key: np.asarray(value).shape
                for key, value in mat.items()
                if not key.startswith("__")
            }
            raise ValueError(
                f"No se encontró una matriz EEG válida en {mat_path}. "
                f"Variables: {available}"
            )

        variable_name, raw = max(candidates, key=lambda item: item[1].size)

    raw = np.asarray(raw, dtype=np.float32)
    if raw.shape[0] == expected_channels:
        eeg = raw
    elif raw.shape[1] == expected_channels:
        eeg = raw.T
    else:
        raise ValueError(f"{subject_id}: forma inesperada {raw.shape}.")

    if not np.isfinite(eeg).all():
        raise ValueError(f"{subject_id}: la señal contiene NaN o Inf.")

    return eeg.astype(np.float32), variable_name


def get_segmented_data(
    folds,
    protocol_subjects,
    adhd_dir=ADHD_DIR,
    control_dir=CONTROL_DIR,
    window_size=512,
    overlap=0.50,
):
    protocol_subjects = set(protocol_subjects)
    adhd_files = build_mat_index(adhd_dir)
    control_files = build_mat_index(control_dir)

    repeated = set(adhd_files) & set(control_files)
    if repeated:
        raise ValueError(f"IDs presentes en ambas clases: {sorted(repeated)}")

    available_subjects = set(adhd_files) | set(control_files)
    missing = protocol_subjects - available_subjects
    if missing:
        raise FileNotFoundError(
            f"Faltan archivos .mat para sujetos de folds.pkl: {sorted(missing)}"
        )

    unused_subjects = sorted(available_subjects - protocol_subjects)
    control_subjects = sorted(protocol_subjects & set(control_files))
    adhd_subjects = sorted(protocol_subjects & set(adhd_files))

    if len(control_subjects) != 60 or len(adhd_subjects) != 60:
        raise ValueError(
            f"La cohorte debe ser 60/60; se obtuvo "
            f"Control={len(control_subjects)}, ADHD={len(adhd_subjects)}."
        )

    records = [
        (subject, 0, "Control", control_files[subject])
        for subject in control_subjects
    ] + [
        (subject, 1, "ADHD", adhd_files[subject])
        for subject in adhd_subjects
    ]

    stride = int(round(window_size * (1.0 - overlap)))
    if stride <= 0:
        raise ValueError("El stride calculado no es válido.")

    windows = []
    labels = []
    subject_ids = []
    window_ids = []
    metadata_rows = []
    global_index = 0

    for subject_id, label, class_name, mat_path in records:
        eeg, variable_name = load_eeg_mat(mat_path, subject_id)
        n_samples = eeg.shape[1]

        if n_samples < window_size:
            raise ValueError(
                f"{subject_id}: {n_samples} muestras, menos de {window_size}."
            )

        local_window_id = 0
        for start_sample in range(0, n_samples - window_size + 1, stride):
            end_sample = start_sample + window_size
            windows.append(eeg[:, start_sample:end_sample])
            labels.append(label)
            subject_ids.append(subject_id)
            window_ids.append(local_window_id)
            metadata_rows.append(
                {
                    "global_window_index": global_index,
                    "subject_id": subject_id,
                    "label": label,
                    "class_name": class_name,
                    "window_id": local_window_id,
                    "window_name": f"Window {local_window_id + 1}",
                    "start_sample": start_sample,
                    "end_sample": end_sample,
                    "n_recording_samples": n_samples,
                    "mat_variable": variable_name,
                    "mat_filename": mat_path.name,
                }
            )
            local_window_id += 1
            global_index += 1

    X = np.stack(windows).astype(np.float32)
    y_binary = np.asarray(labels, dtype=np.int64)
    y_onehot = np.eye(2, dtype=np.float32)[y_binary]
    subject_ids = np.asarray(subject_ids, dtype=str)
    window_ids = np.asarray(window_ids, dtype=np.int64)
    metadata = pd.DataFrame(metadata_rows)

    loaded_subjects = set(subject_ids)
    if loaded_subjects != protocol_subjects:
        raise RuntimeError(
            "Los sujetos cargados no coinciden con folds.pkl. "
            f"Faltantes={sorted(protocol_subjects-loaded_subjects)}, "
            f"extras={sorted(loaded_subjects-protocol_subjects)}"
        )

    if "v28p" not in loaded_subjects:
        raise RuntimeError("v28p no fue cargado.")
    if "v36p" in loaded_subjects:
        raise RuntimeError("v36p fue cargado aunque no pertenece al protocolo.")

    for fold_index, (train_subjects, val_subjects, test_subjects) in enumerate(folds):
        actual_train = set(subject_ids[np.isin(subject_ids, train_subjects)])
        actual_val = set(subject_ids[np.isin(subject_ids, val_subjects)])
        actual_test = set(subject_ids[np.isin(subject_ids, test_subjects)])
        if actual_train != set(train_subjects):
            raise RuntimeError(f"Fold {fold_index}: train incompleto.")
        if actual_val != set(val_subjects):
            raise RuntimeError(f"Fold {fold_index}: validation incompleto.")
        if actual_test != set(test_subjects):
            raise RuntimeError(f"Fold {fold_index}: test incompleto.")
        if len(actual_test) != 24:
            raise RuntimeError(f"Fold {fold_index}: test tiene {len(actual_test)} sujetos.")

    print("=" * 80)
    print("COHORTE CARGADA Y VALIDADA")
    print("=" * 80)
    print("X:", X.shape, X.dtype)
    print("y:", y_onehot.shape, y_onehot.dtype)
    print("Sujetos:", len(loaded_subjects))
    print("Control:", len(control_subjects))
    print("ADHD:", len(adhd_subjects))
    print("v28p incluido:", "v28p" in loaded_subjects)
    print("v36p incluido:", "v36p" in loaded_subjects)
    print("Disponibles pero no usados:", unused_subjects)

    return {
        "X": X,
        "y_binary": y_binary,
        "y_onehot": y_onehot,
        "subject_ids": subject_ids,
        "window_ids": window_ids,
        "metadata": metadata,
        "folds": folds,
        "protocol_subjects": sorted(protocol_subjects),
        "unused_subjects": unused_subjects,
    }


In [3]:
# ============================================================
# 3. EXTRACCIÓN Y SELECCIÓN DE CARACTERÍSTICAS IM-CBGT
# ============================================================

class IMCBGTFeatureExtractor:
    """Extracción equivalente al cuaderno original, vectorizada por bloques."""

    def __init__(self, fs=128, eps=1e-8, chunk_size=256):
        self.fs = fs
        self.eps = eps
        self.chunk_size = chunk_size

    def extract_features(self, X):
        X = np.asarray(X, dtype=np.float32)
        all_features = []

        for start in range(0, len(X), self.chunk_size):
            stop = min(start + self.chunk_size, len(X))
            x = X[start:stop].astype(np.float64, copy=False)

            mean_value = np.mean(x, axis=-1)
            std_value = np.std(x, axis=-1)
            skew_value = skew(x, axis=-1, bias=True, nan_policy="omit")
            kurt_value = kurtosis(x, axis=-1, bias=True, nan_policy="omit")

            dx = np.diff(x, axis=-1)
            ddx = np.diff(dx, axis=-1)
            var_x = np.var(x, axis=-1) + self.eps
            var_dx = np.var(dx, axis=-1) + self.eps
            var_ddx = np.var(ddx, axis=-1) + self.eps
            mobility = np.sqrt(var_dx / var_x)
            complexity = np.sqrt(var_ddx / var_dx) / (mobility + self.eps)

            frequencies, psd = welch(
                x,
                fs=self.fs,
                nperseg=min(256, x.shape[-1]),
                axis=-1,
            )
            psd_sum = np.sum(psd, axis=-1, keepdims=True) + self.eps
            normalized_psd = psd / psd_sum
            spectral_entropy = entropy(normalized_psd, axis=-1)

            bands = [
                (0.5, 4),
                (4, 8),
                (8, 13),
                (13, 30),
                (30, 45),
            ]
            band_powers = []
            for low, high in bands:
                band_mask = (frequencies >= low) & (frequencies <= high)
                band_powers.append(np.sum(psd[..., band_mask], axis=-1))

            # Orden idéntico al cuaderno: por canal y luego las 12 características.
            stacked = np.stack(
                [
                    mean_value,
                    std_value,
                    skew_value,
                    kurt_value,
                    mobility,
                    complexity,
                    *band_powers,
                    spectral_entropy,
                ],
                axis=-1,
            )
            features = stacked.reshape(stacked.shape[0], -1)
            features = np.nan_to_num(
                features,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            ).astype(np.float32)
            all_features.append(features)

            print(
                f"Características: {stop}/{len(X)} ventanas procesadas",
                end="\r",
            )

        print()
        return np.concatenate(all_features, axis=0)


class IMCBGTFeatureSelector:
    def __init__(self, pca_var=0.95, chi_k=100):
        self.scaler = StandardScaler()
        self.pca = PCA(n_components=pca_var)
        self.chi_k = chi_k
        self.chi = None

    def fit_transform(self, X, y_binary):
        X_scaled = self.scaler.fit_transform(X)
        X_pca = self.pca.fit_transform(X_scaled)
        n_features = X_pca.shape[1]
        k = min(self.chi_k, n_features)
        self.chi = SelectKBest(chi2, k="all" if k == n_features else k)
        selected = self.chi.fit_transform(np.abs(X_pca), y_binary)
        return selected.astype(np.float32)

    def transform(self, X):
        X_scaled = self.scaler.transform(X)
        X_pca = self.pca.transform(X_scaled)
        return self.chi.transform(np.abs(X_pca)).astype(np.float32)


def subject_split_indices(subject_ids, selected_subjects):
    selected_subjects = set(normalize_subject_id(x) for x in selected_subjects)
    indices = np.where(np.isin(subject_ids, list(selected_subjects)))[0]
    actual_subjects = set(subject_ids[indices])
    if actual_subjects != selected_subjects:
        raise RuntimeError(
            f"La selección no coincide: faltantes={sorted(selected_subjects-actual_subjects)}"
        )
    return indices.astype(np.int64)


def prepare_fold_features(
    fold_index,
    fold,
    raw_features,
    y_binary,
    subject_ids,
    cache_root,
    pca_var=0.95,
    chi_k=100,
):
    cache_root = Path(cache_root)
    cache_root.mkdir(parents=True, exist_ok=True)
    cache_path = cache_root / f"fold_{fold_index}_features.npz"
    selector_path = cache_root / f"fold_{fold_index}_selector.pkl"

    train_subjects, val_subjects, test_subjects = fold
    train_idx = subject_split_indices(subject_ids, train_subjects)
    val_idx = subject_split_indices(subject_ids, val_subjects)
    test_idx = subject_split_indices(subject_ids, test_subjects)

    if cache_path.exists() and selector_path.exists() and not FORCE_RETRAIN:
        cached = np.load(cache_path)
        return {
            "train_idx": train_idx,
            "val_idx": val_idx,
            "test_idx": test_idx,
            "X_train": cached["X_train"],
            "X_val": cached["X_val"],
            "X_test": cached["X_test"],
            "selector_path": selector_path,
        }

    selector = IMCBGTFeatureSelector(pca_var=pca_var, chi_k=chi_k)
    X_train = selector.fit_transform(raw_features[train_idx], y_binary[train_idx])
    X_val = selector.transform(raw_features[val_idx])
    X_test = selector.transform(raw_features[test_idx])

    # Entrada esperada por el modelo: (N, features, 1)
    X_train = X_train[:, :, None].astype(np.float32)
    X_val = X_val[:, :, None].astype(np.float32)
    X_test = X_test[:, :, None].astype(np.float32)

    np.savez_compressed(
        cache_path,
        X_train=X_train,
        X_val=X_val,
        X_test=X_test,
    )
    with open(selector_path, "wb") as file:
        pickle.dump(selector, file)

    print(
        f"Fold {fold_index}: features train/val/test = "
        f"{X_train.shape}, {X_val.shape}, {X_test.shape}"
    )

    return {
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "selector_path": selector_path,
    }


In [4]:
# ============================================================
# 4. ARQUITECTURA IM-CBGT ORIGINAL EN PYTORCH
# ============================================================

class IMCBGTTorch(nn.Module):
    def __init__(
        self,
        input_features,
        nb_classes=2,
        conv_filters=64,
        kernel_size=3,
        lstm_units=64,
        gru_units=64,
        att_heads=2,
        dense_units=128,
        fusion_units=256,
        dropout_rate=0.5,
    ):
        super().__init__()

        self.input_features = int(input_features)
        if self.input_features < 2:
            raise ValueError("input_features debe ser >= 2.")
        if gru_units % att_heads != 0:
            raise ValueError("gru_units debe ser divisible por att_heads.")

        pooled_len = self.input_features // 2

        self.cnn_branch = nn.Sequential(
            nn.Conv1d(
                in_channels=1,
                out_channels=conv_filters,
                kernel_size=kernel_size,
                padding=kernel_size // 2,
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(dropout_rate),
            nn.Flatten(),
            nn.Linear(conv_filters * pooled_len, dense_units),
            nn.ReLU(),
        )

        self.conv_bilstm = nn.Sequential(
            nn.Conv1d(
                in_channels=1,
                out_channels=conv_filters,
                kernel_size=kernel_size,
                padding=kernel_size // 2,
            ),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
        )
        self.bilstm = nn.LSTM(
            input_size=conv_filters,
            hidden_size=lstm_units,
            batch_first=True,
            bidirectional=True,
        )
        self.bilstm_dense = nn.Sequential(
            nn.Linear(2 * lstm_units, dense_units),
            nn.ReLU(),
        )

        self.gru = nn.GRU(
            input_size=1,
            hidden_size=gru_units,
            batch_first=True,
        )
        self.self_attention = nn.MultiheadAttention(
            embed_dim=gru_units,
            num_heads=att_heads,
            batch_first=True,
        )
        self.layer_norm = nn.LayerNorm(gru_units)
        self.gru_att_dense = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.input_features * gru_units, dense_units),
            nn.ReLU(),
        )

        self.fusion = nn.Sequential(
            nn.Linear(3 * dense_units, fusion_units),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(fusion_units, nb_classes),
        )

    def forward(self, x):
        if x.ndim == 2:
            x = x.unsqueeze(-1)
        x_conv = x.transpose(1, 2)

        x1 = self.cnn_branch(x_conv)

        z2 = self.conv_bilstm(x_conv).transpose(1, 2)
        _, (h_n, _) = self.bilstm(z2)
        h_bilstm = torch.cat([h_n[-2], h_n[-1]], dim=1)
        x2 = self.bilstm_dense(h_bilstm)

        z3, _ = self.gru(x)
        attention_output, _ = self.self_attention(
            z3,
            z3,
            z3,
            need_weights=False,
        )
        z3 = self.layer_norm(z3 + attention_output)
        x3 = self.gru_att_dense(z3)

        merged = torch.cat([x1, x2, x3], dim=1)
        logits = self.fusion(merged)
        probabilities = F.softmax(logits, dim=1)

        return {
            "logits": logits,
            "out_activation": probabilities,
        }


class IMCBGTClassificationLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, outputs, y_true):
        probabilities = torch.clamp(
            outputs["out_activation"],
            self.eps,
            1.0 - self.eps,
        )
        return (-torch.sum(y_true * torch.log(probabilities), dim=1)).mean()


In [5]:
# ============================================================
# 5. ENTRENAMIENTO, EVALUACIÓN Y GUARDADO
# ============================================================

def make_loader(X, y_onehot, batch_size, shuffle, seed=None):
    dataset = TensorDataset(
        torch.from_numpy(np.asarray(X, dtype=np.float32)),
        torch.from_numpy(np.asarray(y_onehot, dtype=np.float32)),
    )
    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(int(seed))

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )


def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True)
        yb = yb.to(DEVICE, dtype=torch.float32, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(xb)
        loss = loss_fn(outputs, yb)
        loss.backward()
        optimizer.step()

        current_batch = xb.size(0)
        total_loss += loss.item() * current_batch
        n_samples += current_batch

    return total_loss / max(n_samples, 1)


@torch.no_grad()
def evaluate_model(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    n_samples = 0
    probability_batches = []
    prediction_batches = []
    true_batches = []

    for xb, yb in loader:
        xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True)
        yb = yb.to(DEVICE, dtype=torch.float32, non_blocking=True)
        outputs = model(xb)
        probabilities = outputs["out_activation"]
        loss = loss_fn(outputs, yb)

        current_batch = xb.size(0)
        total_loss += loss.item() * current_batch
        n_samples += current_batch

        probability_batches.append(probabilities[:, 1].cpu().numpy())
        prediction_batches.append(torch.argmax(probabilities, dim=1).cpu().numpy())
        true_batches.append(torch.argmax(yb, dim=1).cpu().numpy())

    y_prob = np.concatenate(probability_batches).astype(np.float32)
    y_pred = np.concatenate(prediction_batches).astype(np.int64)
    y_true = np.concatenate(true_batches).astype(np.int64)

    try:
        auc_value = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        auc_value = float("nan")

    return {
        "loss": float(total_loss / max(n_samples, 1)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred)),
        "auc": auc_value,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }


def scalar_metrics(metrics):
    return {
        key: float(metrics[key])
        for key in [
            "loss",
            "accuracy",
            "balanced_accuracy",
            "recall",
            "precision",
            "kappa",
            "auc",
        ]
    }


def save_window_and_subject_results(
    seed,
    fold_index,
    test_idx,
    data,
    test_metrics,
    fold_dir,
):
    metadata_test = data["metadata"].iloc[test_idx].reset_index(drop=True)
    subject_ids_test = data["subject_ids"][test_idx]
    window_ids_test = data["window_ids"][test_idx]

    predictions = pd.DataFrame(
        {
            "model": MODEL_NAME,
            "seed": seed,
            "fold": fold_index,
            "global_window_index": metadata_test["global_window_index"].to_numpy(),
            "subject_id": subject_ids_test,
            "label": test_metrics["y_true"],
            "class_name": np.where(test_metrics["y_true"] == 1, "ADHD", "Control"),
            "window_id": window_ids_test,
            "window_name": metadata_test["window_name"].to_numpy(),
            "start_sample": metadata_test["start_sample"].to_numpy(),
            "end_sample": metadata_test["end_sample"].to_numpy(),
            "y_true": test_metrics["y_true"],
            "prob_adhd": test_metrics["y_prob"],
            "y_pred": test_metrics["y_pred"],
            "correct": (test_metrics["y_true"] == test_metrics["y_pred"]).astype(np.int64),
        }
    )

    predictions_path = Path(fold_dir) / "test_window_predictions.csv"
    predictions.to_csv(predictions_path, index=False)

    subject_summary = (
        predictions.groupby(
            ["model", "seed", "fold", "subject_id", "label", "class_name"],
            as_index=False,
        )
        .agg(
            n_windows=("correct", "size"),
            n_correct_windows=("correct", "sum"),
            window_accuracy=("correct", "mean"),
            mean_prob_adhd=("prob_adhd", "mean"),
            std_prob_adhd=("prob_adhd", "std"),
        )
    )

    if subject_summary["subject_id"].nunique() != 24:
        raise RuntimeError(
            f"Fold {fold_index}: el resumen tiene "
            f"{subject_summary['subject_id'].nunique()} sujetos de test."
        )

    subject_summary_path = Path(fold_dir) / "subject_summary.csv"
    subject_summary.to_csv(subject_summary_path, index=False)
    return predictions_path, subject_summary_path


def build_model(input_features):
    return IMCBGTTorch(
        input_features=input_features,
        nb_classes=MODEL_ARGS["nb_classes"],
        conv_filters=MODEL_ARGS["conv_filters"],
        kernel_size=MODEL_ARGS["kernel_size"],
        lstm_units=MODEL_ARGS["lstm_units"],
        gru_units=MODEL_ARGS["gru_units"],
        att_heads=MODEL_ARGS["att_heads"],
        dense_units=MODEL_ARGS["dense_units"],
        fusion_units=MODEL_ARGS["fusion_units"],
        dropout_rate=MODEL_ARGS["dropout_rate"],
    )


def train_one_fold(seed, fold_index, fold_data, data):
    run_dir = SAVE_ROOT / f"IMCBGT_TDAH_fixed_seed_{seed}"
    fold_dir = run_dir / f"fold_{fold_index}"
    fold_dir.mkdir(parents=True, exist_ok=True)

    best_state_path = fold_dir / "best_state.pt"
    final_state_path = fold_dir / "final_state_dict.pt"
    resume_path = fold_dir / "resume_checkpoint.pt"
    fold_result_path = fold_dir / "fold_result.pkl"
    history_path = fold_dir / "history.pkl"
    predictions_path = fold_dir / "test_window_predictions.csv"
    subject_summary_path = fold_dir / "subject_summary.csv"

    if FORCE_RETRAIN and fold_dir.exists():
        for path in fold_dir.iterdir():
            if path.is_file():
                path.unlink()

    if (
        not FORCE_RETRAIN
        and fold_result_path.exists()
        and best_state_path.exists()
        and predictions_path.exists()
        and subject_summary_path.exists()
    ):
        with open(fold_result_path, "rb") as file:
            result = pickle.load(file)
        print(f"seed={seed} fold={fold_index}: reutilizado.")
        return result

    fold_seed = seed + fold_index
    set_seed(fold_seed)

    train_idx = fold_data["train_idx"]
    val_idx = fold_data["val_idx"]
    test_idx = fold_data["test_idx"]
    X_train = fold_data["X_train"]
    X_val = fold_data["X_val"]
    X_test = fold_data["X_test"]
    y_train = data["y_onehot"][train_idx]
    y_val = data["y_onehot"][val_idx]
    y_test = data["y_onehot"][test_idx]

    train_subjects = set(data["subject_ids"][train_idx])
    val_subjects = set(data["subject_ids"][val_idx])
    test_subjects = set(data["subject_ids"][test_idx])
    if (len(train_subjects), len(val_subjects), len(test_subjects)) != (76, 20, 24):
        raise RuntimeError(
            f"seed={seed} fold={fold_index}: conteos de sujetos inválidos."
        )

    train_loader = make_loader(
        X_train,
        y_train,
        BATCH_SIZE,
        shuffle=True,
        seed=fold_seed,
    )
    val_loader = make_loader(X_val, y_val, BATCH_SIZE, shuffle=False)
    test_loader = make_loader(X_test, y_test, BATCH_SIZE, shuffle=False)

    model = build_model(input_features=X_train.shape[1]).to(DEVICE)
    loss_fn = IMCBGTClassificationLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=TRAIN_CONFIG["learning_rate"],
        weight_decay=TRAIN_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=TRAIN_CONFIG["scheduler_factor"],
        patience=TRAIN_CONFIG["scheduler_patience"],
        min_lr=TRAIN_CONFIG["scheduler_min_lr"],
    )

    start_epoch = 0
    best_val_loss = float("inf")
    best_epoch = 0
    patience_counter = 0
    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_balanced_accuracy": [],
        "val_recall": [],
        "val_precision": [],
        "val_kappa": [],
        "val_auc": [],
        "lr": [],
    }

    if not FORCE_RETRAIN and resume_path.exists():
        checkpoint = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        best_val_loss = float(checkpoint["best_val_loss"])
        best_epoch = int(checkpoint["best_epoch"])
        patience_counter = int(checkpoint["patience_counter"])
        history = checkpoint["history"]
        print(
            f"seed={seed} fold={fold_index}: reanudando desde epoch {start_epoch + 1}."
        )

    for epoch in range(start_epoch, EPOCHS):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn)
        val_metrics = evaluate_model(model, val_loader, loss_fn)
        scheduler.step(val_metrics["loss"])

        history["epoch"].append(epoch + 1)
        history["train_loss"].append(float(train_loss))
        history["val_loss"].append(val_metrics["loss"])
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_balanced_accuracy"].append(val_metrics["balanced_accuracy"])
        history["val_recall"].append(val_metrics["recall"])
        history["val_precision"].append(val_metrics["precision"])
        history["val_kappa"].append(val_metrics["kappa"])
        history["val_auc"].append(val_metrics["auc"])
        history["lr"].append(float(optimizer.param_groups[0]["lr"]))

        improved = val_metrics["loss"] < (
            best_val_loss - TRAIN_CONFIG["min_delta"]
        )
        if improved:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), best_state_path)
        else:
            patience_counter += 1

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "history": history,
            },
            resume_path,
        )
        with open(history_path, "wb") as file:
            pickle.dump(history, file)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"seed={seed} fold={fold_index} epoch={epoch+1:03d} | "
                f"train_loss={train_loss:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} | "
                f"val_acc={val_metrics['accuracy']:.4f}"
            )

        if patience_counter >= TRAIN_CONFIG["early_stopping_patience"]:
            print(
                f"seed={seed} fold={fold_index}: early stopping en "
                f"epoch {epoch+1}; mejor epoch={best_epoch}."
            )
            break

    if not best_state_path.exists():
        raise RuntimeError(
            f"seed={seed} fold={fold_index}: no se guardó best_state.pt."
        )

    best_state = torch.load(best_state_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), final_state_path)

    final_val_metrics = evaluate_model(model, val_loader, loss_fn)
    final_test_metrics = evaluate_model(model, test_loader, loss_fn)

    saved_predictions_path, saved_subject_summary_path = save_window_and_subject_results(
        seed=seed,
        fold_index=fold_index,
        test_idx=test_idx,
        data=data,
        test_metrics=final_test_metrics,
        fold_dir=fold_dir,
    )

    result = {
        "model_name": MODEL_NAME,
        "seed": seed,
        "fold": fold_index,
        "best_epoch": best_epoch,
        "fold_metrics": scalar_metrics(final_test_metrics),
        "fold_val_metrics": scalar_metrics(final_val_metrics),
        "history": history,
        "evaluate_test": True,
        "model_args": {
            **MODEL_ARGS,
            "input_features": int(X_train.shape[1]),
        },
        "training_config": TRAIN_CONFIG,
        "best_state_path": str(best_state_path),
        "final_state_path": str(final_state_path),
        "selector_path": str(fold_data["selector_path"]),
        "predictions_path": str(saved_predictions_path),
        "subject_summary_path": str(saved_subject_summary_path),
        "train_subjects": sorted(train_subjects),
        "val_subjects": sorted(val_subjects),
        "test_subjects": sorted(test_subjects),
        "n_train_windows": int(len(train_idx)),
        "n_val_windows": int(len(val_idx)),
        "n_test_windows": int(len(test_idx)),
    }
    with open(fold_result_path, "wb") as file:
        pickle.dump(result, file)

    if resume_path.exists():
        resume_path.unlink()

    print(
        f"seed={seed} fold={fold_index} TEST | "
        f"acc={final_test_metrics['accuracy']:.4f} | "
        f"bacc={final_test_metrics['balanced_accuracy']:.4f} | "
        f"auc={final_test_metrics['auc']:.4f}"
    )

    del model, optimizer, scheduler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


In [6]:
# ============================================================
# 6. CARGAR LOS 120 SUJETOS Y PREPARAR FEATURES UNA SOLA VEZ
# ============================================================

folds, protocol_subjects = load_and_validate_folds(FOLDS_PATH)
data = get_segmented_data(
    folds=folds,
    protocol_subjects=protocol_subjects,
    adhd_dir=ADHD_DIR,
    control_dir=CONTROL_DIR,
    window_size=512,
    overlap=0.50,
)

RAW_FEATURES_PATH = SAVE_ROOT / "imcbgt_raw_features_all_windows.npy"
FEATURE_CACHE_ROOT = SAVE_ROOT / "fold_feature_cache"
FEATURE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

if RAW_FEATURES_PATH.exists() and not FORCE_RETRAIN:
    raw_features = np.load(RAW_FEATURES_PATH)
    if len(raw_features) != len(data["X"]):
        raise RuntimeError("El caché de features no coincide con las ventanas actuales.")
    print("Features crudas reutilizadas:", raw_features.shape)
else:
    extractor = IMCBGTFeatureExtractor(
        fs=MODEL_ARGS["fs"],
        chunk_size=256,
    )
    raw_features = extractor.extract_features(data["X"])
    np.save(RAW_FEATURES_PATH, raw_features)
    print("Features crudas guardadas:", raw_features.shape)

fold_feature_data = {}
for fold_index, fold in enumerate(folds):
    fold_feature_data[fold_index] = prepare_fold_features(
        fold_index=fold_index,
        fold=fold,
        raw_features=raw_features,
        y_binary=data["y_binary"],
        subject_ids=data["subject_ids"],
        cache_root=FEATURE_CACHE_ROOT,
        pca_var=MODEL_ARGS["pca_var"],
        chi_k=MODEL_ARGS["chi_k"],
    )

print("\nPreparación completada para los cinco folds.")


COHORTE CARGADA Y VALIDADA
X: (8213, 19, 512) float32
y: (8213, 2) float32
Sujetos: 120
Control: 60
ADHD: 60
v28p incluido: True
v36p incluido: False
Disponibles pero no usados: ['v36p']
Características: 8213/8213 ventanas procesadas
Features crudas guardadas: (8213, 228)
Fold 0: features train/val/test = (4977, 69, 1), (1574, 69, 1), (1662, 69, 1)
Fold 1: features train/val/test = (5223, 69, 1), (1428, 69, 1), (1562, 69, 1)
Fold 2: features train/val/test = (5211, 69, 1), (1343, 69, 1), (1659, 69, 1)
Fold 3: features train/val/test = (5305, 73, 1), (1300, 73, 1), (1608, 73, 1)
Fold 4: features train/val/test = (5080, 86, 1), (1411, 86, 1), (1722, 86, 1)

Preparación completada para los cinco folds.


In [7]:
# ============================================================
# 7. EJECUTAR 10 SEEDS × 5 FOLDS
# ============================================================

all_fold_results = []

for seed in SEEDS:
    print("\n" + "=" * 90)
    print(f"SEED {seed} — 5 FOLDS")
    print("=" * 90)

    for fold_index in range(N_FOLDS):
        result = train_one_fold(
            seed=seed,
            fold_index=fold_index,
            fold_data=fold_feature_data[fold_index],
            data=data,
        )
        all_fold_results.append(result)

print("\nEntrenamientos completados:", len(all_fold_results))



SEED 0 — 5 FOLDS
seed=0 fold=0 epoch=001 | train_loss=0.6112 | val_loss=0.8330 | val_acc=0.5565
seed=0 fold=0 epoch=005 | train_loss=0.3540 | val_loss=0.9657 | val_acc=0.5947
seed=0 fold=0 epoch=010 | train_loss=0.2523 | val_loss=1.2046 | val_acc=0.5870
seed=0 fold=0 epoch=015 | train_loss=0.1521 | val_loss=1.3914 | val_acc=0.6264
seed=0 fold=0 epoch=020 | train_loss=0.0977 | val_loss=1.7790 | val_acc=0.6283
seed=0 fold=0 epoch=025 | train_loss=0.0594 | val_loss=2.2704 | val_acc=0.6264
seed=0 fold=0: early stopping en epoch 27; mejor epoch=2.
seed=0 fold=0 TEST | acc=0.7034 | bacc=0.6747 | auc=0.8036
seed=0 fold=1 epoch=001 | train_loss=0.6219 | val_loss=0.6020 | val_acc=0.6695
seed=0 fold=1 epoch=005 | train_loss=0.4004 | val_loss=0.6674 | val_acc=0.6681
seed=0 fold=1 epoch=010 | train_loss=0.2743 | val_loss=0.8174 | val_acc=0.6569
seed=0 fold=1 epoch=015 | train_loss=0.1609 | val_loss=1.1414 | val_acc=0.6534
seed=0 fold=1 epoch=020 | train_loss=0.0918 | val_loss=1.5013 | val_acc=0.6

In [8]:
# ============================================================
# 8. CONSOLIDAR MÉTRICAS, PREDICCIONES Y RESULTADOS POR SUJETO
# ============================================================

def summarize_metric(values):
    values = np.asarray(values, dtype=float)
    return {
        "mean": float(np.nanmean(values)),
        "std_population": float(np.nanstd(values, ddof=0)),
        "std_sample": float(np.nanstd(values, ddof=1)) if len(values) > 1 else 0.0,
        "n": int(len(values)),
    }

metric_rows = []
window_frames = []
subject_frames = []

for result in all_fold_results:
    metrics = result["fold_metrics"]
    metric_rows.append(
        {
            "model": MODEL_NAME,
            "seed": result["seed"],
            "fold": result["fold"],
            "best_epoch": result["best_epoch"],
            "n_test_windows": result["n_test_windows"],
            "n_test_subjects": len(result["test_subjects"]),
            "accuracy": metrics["accuracy"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "recall": metrics["recall"],
            "precision": metrics["precision"],
            "kappa": metrics["kappa"],
            "auc": metrics["auc"],
        }
    )
    window_frames.append(pd.read_csv(result["predictions_path"]))
    subject_frames.append(pd.read_csv(result["subject_summary_path"]))

metrics_df = pd.DataFrame(metric_rows)
window_predictions = pd.concat(window_frames, ignore_index=True)
subject_by_seed = pd.concat(subject_frames, ignore_index=True)

if len(metrics_df) != 50:
    raise RuntimeError(f"Se esperaban 50 resultados y se obtuvieron {len(metrics_df)}.")
if not (metrics_df["n_test_subjects"] == 24).all():
    raise RuntimeError("Hay folds con menos de 24 sujetos de test.")
if len(subject_by_seed) != 1200:
    raise RuntimeError(
        f"Se esperaban 1200 filas sujeto/seed y se obtuvieron {len(subject_by_seed)}."
    )

subject_across_seeds = (
    subject_by_seed.groupby(
        ["model", "subject_id", "label", "class_name"],
        as_index=False,
    )
    .agg(
        fold=("fold", "first"),
        n_seeds=("seed", "nunique"),
        n_windows_per_seed=("n_windows", "first"),
        mean_window_accuracy=("window_accuracy", "mean"),
        std_window_accuracy=("window_accuracy", "std"),
        median_window_accuracy=("window_accuracy", "median"),
        min_window_accuracy=("window_accuracy", "min"),
        max_window_accuracy=("window_accuracy", "max"),
        mean_prob_adhd=("mean_prob_adhd", "mean"),
    )
)
subject_across_seeds["mean_window_accuracy_percent"] = (
    100.0 * subject_across_seeds["mean_window_accuracy"]
)
subject_across_seeds["std_window_accuracy_percent"] = (
    100.0 * subject_across_seeds["std_window_accuracy"]
)

if subject_across_seeds["subject_id"].nunique() != 120:
    raise RuntimeError("El resumen final no contiene 120 sujetos.")
if not (subject_across_seeds["n_seeds"] == 10).all():
    raise RuntimeError("No todos los sujetos tienen diez seeds.")
if "v28p" not in set(subject_across_seeds["subject_id"]):
    raise RuntimeError("v28p no aparece en el resumen final.")
if "v36p" in set(subject_across_seeds["subject_id"]):
    raise RuntimeError("v36p aparece en el resumen final.")

METRICS_PATH = SAVE_ROOT / "repeated_test_results.csv"
WINDOW_PATH = SAVE_ROOT / "IMCBGT_all_window_predictions.csv"
SUBJECT_SEED_PATH = SAVE_ROOT / "IMCBGT_subject_summary_by_seed.csv"
SUBJECT_PATH = SAVE_ROOT / "IMCBGT_subject_accuracy_across_seeds.csv"
SEED_METRICS_PATH = SAVE_ROOT / "IMCBGT_seed_metrics_mean_of_folds.csv"
SUMMARY_JSON_PATH = SAVE_ROOT / "repeated_test_summary.json"

metrics_df.to_csv(METRICS_PATH, index=False)
window_predictions.to_csv(WINDOW_PATH, index=False)
subject_by_seed.to_csv(SUBJECT_SEED_PATH, index=False)
subject_across_seeds.to_csv(SUBJECT_PATH, index=False)

metric_names = [
    "accuracy",
    "balanced_accuracy",
    "recall",
    "precision",
    "kappa",
    "auc",
]
seed_metrics = metrics_df.groupby("seed", as_index=False)[metric_names].mean()
seed_metrics.to_csv(SEED_METRICS_PATH, index=False)

summary_50 = {
    metric: summarize_metric(metrics_df[metric].to_numpy())
    for metric in metric_names
}
summary_10 = {
    metric: summarize_metric(seed_metrics[metric].to_numpy())
    for metric in metric_names
}

summary = {
    "model_name": MODEL_NAME,
    "database": DATABASE_NAME,
    "model_args": MODEL_ARGS,
    "training_config": TRAIN_CONFIG,
    "cohort": {
        "n_subjects": 120,
        "n_control": 60,
        "n_adhd": 60,
        "v28p_included": True,
        "v36p_included": False,
        "unused_subjects": data["unused_subjects"],
    },
    "n_folds": 5,
    "n_seeds": 10,
    "total_test_runs": 50,
    "summary_over_50_folds": summary_50,
    "summary_across_10_seed_means": summary_10,
    "files": {
        "fold_metrics": str(METRICS_PATH),
        "window_predictions": str(WINDOW_PATH),
        "subject_by_seed": str(SUBJECT_SEED_PATH),
        "subject_across_seeds": str(SUBJECT_PATH),
        "seed_metrics": str(SEED_METRICS_PATH),
    },
}
with open(SUMMARY_JSON_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2, ensure_ascii=False)

print("=" * 90)
print("RESULTADOS FINALES IM-CBGT")
print("=" * 90)
print(
    "Accuracy (50 folds):",
    f"{summary_50['accuracy']['mean']:.4f} ± "
    f"{summary_50['accuracy']['std_sample']:.4f}",
)
print(
    "Accuracy (10 medias por seed):",
    f"{summary_10['accuracy']['mean']:.4f} ± "
    f"{summary_10['accuracy']['std_sample']:.4f}",
)
print("v28p incluido:", "v28p" in set(subject_across_seeds["subject_id"]))
print("v36p incluido:", "v36p" in set(subject_across_seeds["subject_id"]))
print("Resultados:", SAVE_ROOT)

# Archivo ZIP opcional para descargar desde Kaggle.
ZIP_BASE = "/kaggle/working/resultados_imcbgt_tdah_ARTICULO_120subjects"
zip_path = shutil.make_archive(ZIP_BASE, "zip", root_dir=SAVE_ROOT)
print("ZIP:", zip_path)


RESULTADOS FINALES IM-CBGT
Accuracy (50 folds): 0.6631 ± 0.0476
Accuracy (10 medias por seed): 0.6631 ± 0.0115
v28p incluido: True
v36p incluido: False
Resultados: /kaggle/working/resultados_imcbgt_tdah_ARTICULO_120subjects
ZIP: /kaggle/working/resultados_imcbgt_tdah_ARTICULO_120subjects.zip


## Reviewer 1 - Comments 1 and 6: participant-level evaluation

This section aggregates the out-of-fold window probabilities within each participant using the arithmetic mean and applies a threshold of 0.5 to obtain the participant-level decision. It reports accuracy, sensitivity, specificity, precision, F1-score, and ROC-AUC.

The 95% confidence intervals are estimated using 5,000 stratified bootstrap resamples with the participant as the sampling unit. The same resampled participants are used across all ten training seeds within each bootstrap replicate, and the metric is then averaged across seeds. Therefore, seeds quantify model-initialization variability and are not treated as independent clinical observations.

The final helper `exact_mcnemar_subject_level` supports paired participant-level comparisons after equivalent out-of-fold predictions are available for each baseline. Holm correction must be applied across the resulting model comparisons.


In [9]:
"""
Subject-level evaluation requested in Reviewer 1, Comments 1 and 6.

This cell must be executed after the reinference cell. It uses only out-of-fold
test predictions and treats the participant, rather than the random seed, as
the sampling unit for the 95% confidence intervals.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score


SUBJECT_PROBABILITY_THRESHOLD = 0.5
N_SUBJECT_BOOTSTRAPS = 5000
SUBJECT_BOOTSTRAP_SEED = 20260826

SUBJECT_METRICS = (
    "accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1_score",
    "roc_auc",
)

SUBJECT_METRIC_LABELS = {
    "accuracy": "Accuracy",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "precision": "Precision",
    "f1_score": "F1-score",
    "roc_auc": "ROC-AUC",
}


def calculate_subject_metrics(y_true, y_prob, threshold=0.5):
    """Calculate the six metrics requested by the reviewer."""
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= threshold).astype(np.int64)

    if y_true.shape != y_prob.shape:
        raise ValueError("y_true and y_prob must have the same shape.")
    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("Subject labels must be binary (0=Control, 1=ADHD).")
    if not np.isfinite(y_prob).all():
        raise ValueError("Subject probabilities contain NaN or Inf values.")

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    def safe_divide(numerator, denominator):
        return float(numerator / denominator) if denominator else np.nan

    sensitivity = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    precision = safe_divide(tp, tp + fp)
    f1_score = safe_divide(
        2.0 * precision * sensitivity,
        precision + sensitivity,
    )

    try:
        roc_auc = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        roc_auc = np.nan

    return {
        "accuracy": safe_divide(tp + tn, len(y_true)),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1_score,
        "roc_auc": roc_auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def prepare_subject_predictions(subject_by_seed, expected_seeds=10):
    """
    Build one out-of-fold prediction per participant and seed.

    The participant probability is the arithmetic mean of all window-level
    ADHD probabilities belonging to that participant. The 0.5 threshold is
    applied only after this within-participant aggregation.
    """
    required_columns = {
        "model",
        "seed",
        "fold",
        "subject_id",
        "label",
        "class_name",
        "n_windows",
        "mean_prob_adhd",
    }
    missing = required_columns - set(subject_by_seed.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    subject_predictions = subject_by_seed[
        [
            "model",
            "seed",
            "fold",
            "subject_id",
            "label",
            "class_name",
            "n_windows",
            "mean_prob_adhd",
        ]
    ].copy()

    subject_predictions = subject_predictions.rename(
        columns={"mean_prob_adhd": "subject_prob_adhd"}
    )
    subject_predictions["subject_id"] = (
        subject_predictions["subject_id"].astype(str)
    )
    subject_predictions["label"] = (
        subject_predictions["label"].astype(int)
    )
    subject_predictions["seed"] = (
        subject_predictions["seed"].astype(int)
    )
    subject_predictions["fold"] = (
        subject_predictions["fold"].astype(int)
    )

    duplicated = subject_predictions.duplicated(
        subset=["model", "seed", "subject_id"],
        keep=False,
    )
    if duplicated.any():
        duplicate_rows = subject_predictions.loc[
            duplicated, ["model", "seed", "subject_id"]
        ]
        raise RuntimeError(
            "Expected exactly one row per participant and seed. Duplicates:\n"
            f"{duplicate_rows.to_string(index=False)}"
        )

    label_counts = subject_predictions.groupby("subject_id")["label"].nunique()
    if not (label_counts == 1).all():
        raise RuntimeError("At least one participant has inconsistent labels.")

    fold_counts = subject_predictions.groupby("subject_id")["fold"].nunique()
    if not (fold_counts == 1).all():
        raise RuntimeError(
            "At least one participant appears in more than one out-of-fold test fold."
        )

    seeds_per_subject = subject_predictions.groupby("subject_id")["seed"].nunique()
    if not (seeds_per_subject == expected_seeds).all():
        raise RuntimeError(
            "Every participant must have one out-of-fold prediction for each seed. "
            f"Observed seed counts: {seeds_per_subject.value_counts().to_dict()}"
        )

    n_subjects = subject_predictions["subject_id"].nunique()
    class_counts = (
        subject_predictions.drop_duplicates("subject_id")["label"].value_counts()
    )
    if n_subjects != 120 or class_counts.to_dict() != {0: 60, 1: 60}:
        raise RuntimeError(
            "Expected 120 participants (60 Control and 60 ADHD), but obtained "
            f"n={n_subjects}, classes={class_counts.to_dict()}."
        )

    subject_predictions["subject_pred"] = (
        subject_predictions["subject_prob_adhd"]
        >= SUBJECT_PROBABILITY_THRESHOLD
    ).astype(np.int64)
    subject_predictions["subject_correct"] = (
        subject_predictions["subject_pred"]
        == subject_predictions["label"]
    ).astype(np.int64)

    return subject_predictions.sort_values(
        ["seed", "fold", "subject_id"]
    ).reset_index(drop=True)


def metrics_for_each_seed(subject_predictions):
    """
    Descriptive model-initialization variability.

    These ten rows must not be used as ten independent clinical samples.
    """
    rows = []
    for seed, seed_frame in subject_predictions.groupby("seed", sort=True):
        seed_frame = seed_frame.sort_values("subject_id")
        metrics = calculate_subject_metrics(
            y_true=seed_frame["label"].to_numpy(),
            y_prob=seed_frame["subject_prob_adhd"].to_numpy(),
            threshold=SUBJECT_PROBABILITY_THRESHOLD,
        )
        rows.append(
            {
                "seed": int(seed),
                "n_subjects": int(len(seed_frame)),
                **metrics,
            }
        )
    return pd.DataFrame(rows)


def participant_stratified_bootstrap(
    subject_predictions,
    n_bootstrap=5000,
    random_state=20260826,
):
    """
    Estimate 95% CIs with participants as the resampling units.

    Within every bootstrap replicate, Control and ADHD participants are
    resampled separately with replacement. The same resampled participants are
    then applied to all ten seeds, metrics are calculated within each seed, and
    the ten metric values are averaged. Therefore, seeds are repeated model
    fits, not independent clinical observations.
    """
    probability_matrix = subject_predictions.pivot(
        index="subject_id",
        columns="seed",
        values="subject_prob_adhd",
    ).sort_index()

    labels = (
        subject_predictions.drop_duplicates("subject_id")
        .set_index("subject_id")["label"]
        .reindex(probability_matrix.index)
        .to_numpy(dtype=np.int64)
    )
    probabilities = probability_matrix.to_numpy(dtype=np.float64)
    seeds = probability_matrix.columns.to_numpy(dtype=int)

    if np.isnan(probabilities).any():
        raise RuntimeError("The participant-by-seed probability matrix is incomplete.")

    control_indices = np.flatnonzero(labels == 0)
    adhd_indices = np.flatnonzero(labels == 1)
    rng = np.random.default_rng(random_state)

    seed_metric_rows = []
    for seed_position, seed in enumerate(seeds):
        metrics = calculate_subject_metrics(
            labels,
            probabilities[:, seed_position],
            threshold=SUBJECT_PROBABILITY_THRESHOLD,
        )
        seed_metric_rows.append(
            {"seed": int(seed), **{name: metrics[name] for name in SUBJECT_METRICS}}
        )
    seed_metric_frame = pd.DataFrame(seed_metric_rows)

    point_estimates = {
        name: float(seed_metric_frame[name].mean())
        for name in SUBJECT_METRICS
    }

    bootstrap_values = np.empty(
        (n_bootstrap, len(SUBJECT_METRICS)),
        dtype=np.float64,
    )

    for bootstrap_index in range(n_bootstrap):
        sampled_indices = np.concatenate(
            [
                rng.choice(
                    control_indices,
                    size=len(control_indices),
                    replace=True,
                ),
                rng.choice(
                    adhd_indices,
                    size=len(adhd_indices),
                    replace=True,
                ),
            ]
        )
        sampled_labels = labels[sampled_indices]

        replicate_seed_metrics = []
        for seed_position in range(probabilities.shape[1]):
            metrics = calculate_subject_metrics(
                sampled_labels,
                probabilities[sampled_indices, seed_position],
                threshold=SUBJECT_PROBABILITY_THRESHOLD,
            )
            replicate_seed_metrics.append(
                [metrics[name] for name in SUBJECT_METRICS]
            )

        bootstrap_values[bootstrap_index] = np.nanmean(
            np.asarray(replicate_seed_metrics, dtype=np.float64),
            axis=0,
        )

    lower = np.nanpercentile(bootstrap_values, 2.5, axis=0)
    upper = np.nanpercentile(bootstrap_values, 97.5, axis=0)

    summary_rows = []
    for metric_position, metric in enumerate(SUBJECT_METRICS):
        estimate = point_estimates[metric]
        ci_lower = float(lower[metric_position])
        ci_upper = float(upper[metric_position])
        summary_rows.append(
            {
                "metric": metric,
                "metric_label": SUBJECT_METRIC_LABELS[metric],
                "estimate": estimate,
                "ci_95_lower": ci_lower,
                "ci_95_upper": ci_upper,
                "estimate_percent": 100.0 * estimate,
                "ci_95_lower_percent": 100.0 * ci_lower,
                "ci_95_upper_percent": 100.0 * ci_upper,
                "n_subjects": int(len(labels)),
                "n_control": int(len(control_indices)),
                "n_adhd": int(len(adhd_indices)),
                "n_seeds": int(len(seeds)),
                "n_bootstrap": int(n_bootstrap),
                "sampling_unit": "participant",
                "bootstrap_type": "stratified percentile bootstrap",
            }
        )

    bootstrap_frame = pd.DataFrame(
        bootstrap_values,
        columns=SUBJECT_METRICS,
    )
    bootstrap_frame.insert(
        0,
        "bootstrap_replicate",
        np.arange(1, n_bootstrap + 1),
    )

    return pd.DataFrame(summary_rows), bootstrap_frame, seed_metric_frame


def build_consensus_subject_predictions(subject_predictions):
    """Create one seed-averaged probability per participant for inspection."""
    consensus = (
        subject_predictions.groupby(
            ["model", "subject_id", "label", "class_name"],
            as_index=False,
        )
        .agg(
            fold=("fold", "first"),
            n_seeds=("seed", "nunique"),
            n_windows=("n_windows", "first"),
            mean_prob_adhd_across_seeds=("subject_prob_adhd", "mean"),
            std_prob_adhd_across_seeds=("subject_prob_adhd", "std"),
        )
    )
    consensus["consensus_pred"] = (
        consensus["mean_prob_adhd_across_seeds"]
        >= SUBJECT_PROBABILITY_THRESHOLD
    ).astype(np.int64)
    consensus["consensus_correct"] = (
        consensus["consensus_pred"] == consensus["label"]
    ).astype(np.int64)
    return consensus.sort_values(["label", "subject_id"]).reset_index(drop=True)


def exact_mcnemar_subject_level(model_a, model_b, name_a="model_a", name_b="model_b"):
    """
    Exact paired McNemar test for two models' consensus subject decisions.

    Each input must contain subject_id, label, and consensus_pred. This helper
    is ready for use after equivalent out-of-fold predictions are exported for
    each baseline. Holm correction should then be applied across comparisons.
    """
    required = {"subject_id", "label", "consensus_pred"}
    for name, frame in ((name_a, model_a), (name_b, model_b)):
        missing = required - set(frame.columns)
        if missing:
            raise KeyError(f"{name} is missing columns: {sorted(missing)}")

    paired = model_a[list(required)].merge(
        model_b[list(required)],
        on="subject_id",
        suffixes=(f"_{name_a}", f"_{name_b}"),
        validate="one_to_one",
    )
    if not np.array_equal(
        paired[f"label_{name_a}"].to_numpy(),
        paired[f"label_{name_b}"].to_numpy(),
    ):
        raise RuntimeError("The paired models contain inconsistent labels.")

    labels = paired[f"label_{name_a}"].to_numpy(dtype=int)
    correct_a = paired[f"consensus_pred_{name_a}"].to_numpy(dtype=int) == labels
    correct_b = paired[f"consensus_pred_{name_b}"].to_numpy(dtype=int) == labels
    a_only = int(np.sum(correct_a & ~correct_b))
    b_only = int(np.sum(~correct_a & correct_b))
    discordant = a_only + b_only
    p_value = (
        float(binomtest(a_only, discordant, p=0.5).pvalue)
        if discordant > 0
        else 1.0
    )
    return {
        "model_a": name_a,
        "model_b": name_b,
        "n_subjects": int(len(paired)),
        "a_correct_b_incorrect": a_only,
        "a_incorrect_b_correct": b_only,
        "exact_mcnemar_p": p_value,
    }


def run_subject_level_review_analysis(
    output_root,
    n_bootstrap=N_SUBJECT_BOOTSTRAPS,
    random_state=SUBJECT_BOOTSTRAP_SEED,
):
    output_root = Path(output_root)
    input_path = output_root / "IMCBGT_subject_summary_by_seed.csv"
    if not input_path.exists():
        raise FileNotFoundError(
            "Run the complete reinference cell first. Missing file:\n"
            f"{input_path}"
        )

    subject_by_seed = pd.read_csv(input_path)
    subject_predictions = prepare_subject_predictions(subject_by_seed)
    descriptive_seed_metrics = metrics_for_each_seed(subject_predictions)
    metric_summary, bootstrap_distribution, bootstrap_seed_metrics = (
        participant_stratified_bootstrap(
            subject_predictions,
            n_bootstrap=n_bootstrap,
            random_state=random_state,
        )
    )
    consensus_predictions = build_consensus_subject_predictions(
        subject_predictions
    )

    manuscript_table = metric_summary[
        [
            "metric_label",
            "estimate_percent",
            "ci_95_lower_percent",
            "ci_95_upper_percent",
        ]
    ].copy()
    manuscript_table["Result (95\\% CI), \\%"] = manuscript_table.apply(
        lambda row: (
            f"{row['estimate_percent']:.1f} "
            f"({row['ci_95_lower_percent']:.1f}--"
            f"{row['ci_95_upper_percent']:.1f})"
        ),
        axis=1,
    )
    manuscript_table = manuscript_table[
        ["metric_label", "Result (95\\% CI), \\%"]
    ].rename(columns={"metric_label": "Metric"})

    output_paths = {
        "subject_predictions_by_seed": (
            output_root / "IMCBGT_subject_level_predictions_by_seed.csv"
        ),
        "subject_metrics_by_seed": (
            output_root / "IMCBGT_subject_level_metrics_by_seed.csv"
        ),
        "subject_metrics_95ci": (
            output_root / "IMCBGT_subject_level_metrics_95CI.csv"
        ),
        "bootstrap_distribution": (
            output_root / "IMCBGT_subject_level_bootstrap_distribution.csv"
        ),
        "bootstrap_seed_metrics": (
            output_root / "IMCBGT_subject_level_point_metrics_by_seed.csv"
        ),
        "consensus_predictions": (
            output_root / "IMCBGT_subject_level_consensus_predictions.csv"
        ),
        "manuscript_table_csv": (
            output_root / "IMCBGT_subject_level_table_for_manuscript.csv"
        ),
        "manuscript_table_latex": (
            output_root / "IMCBGT_subject_level_table_for_manuscript.tex"
        ),
    }

    subject_predictions.to_csv(
        output_paths["subject_predictions_by_seed"], index=False
    )
    descriptive_seed_metrics.to_csv(
        output_paths["subject_metrics_by_seed"], index=False
    )
    metric_summary.to_csv(output_paths["subject_metrics_95ci"], index=False)
    bootstrap_distribution.to_csv(
        output_paths["bootstrap_distribution"], index=False
    )
    bootstrap_seed_metrics.to_csv(
        output_paths["bootstrap_seed_metrics"], index=False
    )
    consensus_predictions.to_csv(
        output_paths["consensus_predictions"], index=False
    )
    manuscript_table.to_csv(
        output_paths["manuscript_table_csv"], index=False
    )
    latex_lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        (
            r"\caption{Participant-level classification performance of "
            r"IMC-BGT. Values are averaged across ten random training seeds; "
            r"95\% confidence intervals were estimated using 5,000 "
            r"stratified bootstrap resamples at the participant level.}"
        ),
        r"\label{tab:subject_level_performance}",
        r"\begin{tabular}{lc}",
        r"\hline",
        r"Metric & Result (95\% CI), \% \\",
        r"\hline",
    ]
    for _, row in manuscript_table.iterrows():
        latex_lines.append(
            f"{row['Metric']} & {row['Result (95\\% CI), \\%']} \\\\"
        )
    latex_lines.extend(
        [
            r"\hline",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )
    output_paths["manuscript_table_latex"].write_text(
        "\n".join(latex_lines) + "\n",
        encoding="utf-8",
    )

    print("\n" + "=" * 80)
    print("REVIEWER 1 - COMMENTS 1 AND 6: SUBJECT-LEVEL RESULTS")
    print("=" * 80)
    print(
        "Aggregation: arithmetic mean of window probabilities within each "
        "participant and seed; threshold = 0.5."
    )
    print(
        "Inference: 95% stratified percentile bootstrap with the participant "
        "as the resampling unit."
    )
    print(
        "The ten seeds quantify model-initialization variability and are not "
        "treated as independent clinical observations.\n"
    )
    print(manuscript_table.to_string(index=False))
    print("\nFiles saved in:", output_root)

    return {
        "subject_predictions_by_seed": subject_predictions,
        "subject_metrics_by_seed": descriptive_seed_metrics,
        "subject_metrics_95ci": metric_summary,
        "bootstrap_distribution": bootstrap_distribution,
        "consensus_predictions": consensus_predictions,
        "manuscript_table": manuscript_table,
        "output_paths": output_paths,
    }


subject_level_results = run_subject_level_review_analysis(
    output_root=SAVE_ROOT,
    n_bootstrap=N_SUBJECT_BOOTSTRAPS,
    random_state=SUBJECT_BOOTSTRAP_SEED,
)



REVIEWER 1 - COMMENTS 1 AND 6: SUBJECT-LEVEL RESULTS
Aggregation: arithmetic mean of window probabilities within each participant and seed; threshold = 0.5.
Inference: 95% stratified percentile bootstrap with the participant as the resampling unit.
The ten seeds quantify model-initialization variability and are not treated as independent clinical observations.

     Metric Result (95\% CI), \%
   Accuracy    72.7 (66.3--78.8)
Sensitivity    85.3 (77.0--92.3)
Specificity    60.0 (50.0--70.0)
  Precision    68.3 (62.7--74.4)
   F1-score    75.8 (70.1--81.0)
    ROC-AUC    78.6 (70.2--86.2)

Files saved in: /kaggle/working/resultados_imcbgt_tdah_ARTICULO_120subjects
